# Telco Customer Churn

#### **Context**
Predict behavior to retain customers. You can analyze all relevant customer data and develop focused customer retention programs.

**Churn**, or customer attrition, is the rate at which customers stop doing business with a company, often measured as a percentage of subscribers who cancel or don't renew over a specific time (monthly, quarterly, yearly)

**Objectives:**

- What's the % of Churn Customers and customers that keep in with the active services?
- Is there any patterns in Churn Customers based on the gender?
- Is there any patterns/preference in Churn Customers based on the type of service provided?
- What's the most profitable service types?
- Which features and services are most profitable?

#### **Content**
Each row represents a customer, each column contains customer’s attributes described on the column Metadata.

**The data set includes information about:**

- **Customers who left within the last month** – the column is called **Churn**
- **Services that each customer has signed up for** – phone, multiple lines, internet, online security, online backup, device protection, tech support, and streaming TV and movies
- **Customer account information** – how long they’ve been a customer, contract, payment method, paperless billing, monthly charges, and total charges
- **Demographic info about customers** – gender, age range, and if they have partners and dependents

## Step 1: Imports packages & dataset

In [ ]:
# Import packages

# For data manipulation
import numpy as np
import pandas as pd

# For data visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter

# For displaying all of the columns in dataframes
pd.set_option('display.max_columns', None)

# For data modeling
from xgboost import XGBClassifier
from xgboost import XGBRegressor
from xgboost import plot_importance

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# For metrics and helpful functions
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.tree import plot_tree

# For saving models
import pickle

In [ ]:
# Download dataset from Kagglehub
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path)

In [ ]:
# Check the dataset files
all_files = os.listdir(path)
print(f"All files: {all_files}")

In [ ]:
# Load dataset into a dataframe
df0 = pd.read_csv('/home/codespace/.cache/kagglehub/datasets/blastchar/telco-customer-churn/versions/1/WA_Fn-UseC_-Telco-Customer-Churn.csv')

## Step 2. Initial EDA and data cleaning

#### Gather basic information about the data

In [ ]:
# Display first few rows of the dataframe
df0.head()

In [ ]:
# Basic information about the data
df0.info()

In [ ]:
# Change datatype of TotalCharges to numeric type
df0['TotalCharges'] = pd.to_numeric(df0['TotalCharges'], errors='coerce')

#### Gather descriptive statistics about the data

In [ ]:
df0.describe()

#### Rename columns

In [ ]:
# Rename columns 
df0 = df0.rename(columns = {'customerID' : 'CustomerID',
                            'gender': 'Gender',
                            'tenure': 'Tenure'})

In [ ]:
# Display all column names after the update
df0.columns

#### Check missing value

In [ ]:
# Check missing value 
df0.isna().sum()

There are 11 missing values in the TotalCharges column

In [ ]:
# Inspect rows containing missing values
df0[np.isnan(df0['TotalCharges'])]

Tenure column is 0 for these entries. We can normally drop these rows.

In [ ]:
# Drop missing values in TotalCharges column
df0 = df0.dropna(subset = 'TotalCharges')

#### Check duplicates

In [ ]:
df0['CustomerID'].duplicated().sum()

There are no duplicates values in the data.

#### Check outliers

In [ ]:
# Create a boxplot to visualize distribution of 'Tenure' and detect any outliers
plt.figure(figsize = (5,3))
plt.title('Distribution of Tenure')
sns.boxplot(x=df0['Tenure'])
plt.show()

There is so outliers in Tenure

In [ ]:
# Create a box plot to visualize distribution of MonthlyCharges and detect any outliers
plt.figure(figsize = (5,3))
plt.title('Distribution of MonthlyCharges')
sns.boxplot(x=df0['MonthlyCharges'])
plt.show()

There is no outliers in Monthly Charges

In [ ]:
# Create a box plot to visualize distribution of TotalCharges and detect any outliers
plt.figure(figsize = (5,3))
plt.title('Distribution of TotalCharges')
sns.boxplot(x=df0['TotalCharges'])
plt.show()

## Step 3: Continue EDA & Visualizations

In [ ]:
df0.head()

#### Churn Rate by Tenure Group

In [ ]:
df1 = df0.copy()

In [ ]:
# Group Tenure by range
bins = [0, 12, 24, 48, 72]
labels = ['0-12', '13-24', '25-48', '49+']

df0['tenure_group'] = pd.cut(
    df0['tenure'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [ ]:
# Function to create charts

def visualize_churn_rate(df, variable):

    # Calculate churn rate by group
    churn_rate = (
        df
        .groupby([variable, 'Churn'], observed=True)
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    
    churn_rate['churn_rate'] = churn_rate['Yes'] / (churn_rate['Yes'] + churn_rate['No'])

    # Set up subplots: 1 row, 2 columns
    fig, axes = plt.subplots(1, 2, figsize=(10,4))

    # Bar chart for Churn Rate
    axes[0].bar(churn_rate[variable], churn_rate['churn_rate'], color=pair_palette[0])
    axes[0].set_xlabel(variable)
    axes[0].set_ylabel('Churn Rate')
    axes[0].set_title(f'Churn Rate by {variable}')

    # Bar chart for Distribution (count)
    total_count = churn_rate['Yes'] + churn_rate['No']
    axes[1].bar(churn_rate[variable], total_count, color=pair_palette[1])
    axes[1].set_xlabel(variable)
    axes[1].set_ylabel('Number of Customers')
    axes[1].set_title(f'Distribution of {variable}')

    plt.tight_layout()
    plt.show()

In [ ]:
visualize_churn_rate(df1,'Contract')

In [ ]:
qualitative_palette = sns.color_palette("Set2")

In [ ]:
pair_palette = sns.color_palette("Paired")